# Lab 07: Investment and Inflation
**Question:** In the 1970s, did investment in stocks beat inflation in the United States?

**Data**
- Inflation: CPI for All Urban Consumers (`CPIAUCSL`, monthly), from FRED.
- Stocks: S&P 500 index (`^GSPC`, monthly close), from Yahoo Finance.
- Dividends (extension): Robert Shiller's monthly S&P 500 data (GitHub mirror), because `^GSPC` is a price index and leaves out dividends.

**Window:** Dec 1969 to Dec 1979, so we capture the full ten years of the 1970s (Jan 1970 through Dec 1979).

In [ ]:
# Step 1: imports
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
# Step 2: CPI from FRED (no API key needed for the CSV link)
cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL",
                  index_col=0, parse_dates=True)
cpi.columns = ["CPI"]
display(cpi.head())
display(cpi.tail())

In [ ]:
# Step 3: S&P 500 from Yahoo Finance
# Yahoo's monthly ^GSPC history only starts in 1985, so pull daily closes
# and keep the last close of each month.
sp_daily = yf.download("^GSPC", start="1969-12-01", end="1980-01-31",
                       interval="1d", auto_adjust=False, progress=False)["Close"]
sp_daily = sp_daily.squeeze().rename("SP500")   # one clean Series, works across yfinance versions
display(sp_daily.head())
display(sp_daily.tail())

In [ ]:
# Step 3b: month-end close
sp = sp_daily.resample("ME").last().to_frame()
display(sp.head())
display(sp.tail())

In [ ]:
# Step 4: put both on a monthly index so concat lines up rows by month
cpi.index = cpi.index.to_period("M")
sp.index = sp.index.to_period("M")

df = pd.concat([sp, cpi], axis=1).loc["1969-12":"1979-12"]
display(df.head())
display(df.tail())
print("Rows:", len(df), "(expect 121)")
print("Missing values:\n", df.isna().sum())   # concat fills NaN on mismatched labels, so check

In [ ]:
# Step 5: index both series to 100 in Dec 1969 and plot
growth = df / df.iloc[0] * 100
display(growth.tail())

growth.plot(figsize=(9, 4), title="S&P 500 (price) vs CPI, Dec 1969 = 100")
plt.ylabel("Index level")
plt.show()

In [ ]:
# Step 6: total and annualized change over the decade
years = 10
total = df.iloc[-1] / df.iloc[0] - 1
annual = (df.iloc[-1] / df.iloc[0]) ** (1 / years) - 1
summary = pd.DataFrame({"Total change": total, "Annualized": annual})
display(summary.style.format("{:.2%}"))

# Real (inflation-adjusted) return: (1 + nominal) / (1 + inflation) - 1
real_total = (1 + total["SP500"]) / (1 + total["CPI"]) - 1
real_annual = (1 + annual["SP500"]) / (1 + annual["CPI"]) - 1
print(f"Real S&P 500 price return, total:      {real_total:.2%}")
print(f"Real S&P 500 price return, annualized: {real_annual:.2%}")

In [ ]:
# Step 7: real S&P 500 level (deflated by CPI), Dec 1969 = 100
df["Real_SP500"] = df["SP500"] / df["CPI"]
df["Real_SP500"] = df["Real_SP500"] / df["Real_SP500"].iloc[0] * 100
display(df.tail())

df["Real_SP500"].plot(figsize=(9, 4), title="Real S&P 500 (price), Dec 1969 = 100")
plt.axhline(100, color="gray", linestyle="--")
plt.ylabel("Index level")
plt.show()

In [ ]:
# Step 8: year by year, did stocks beat inflation?
dec = df.loc[df.index.month == 12, ["SP500", "CPI"]]   # December values, 1969-1979
yearly = dec.pct_change().dropna()
yearly.columns = ["SP500_return", "Inflation"]
yearly["Beat_inflation"] = yearly["SP500_return"] > yearly["Inflation"]
display(yearly)
print("Years stocks (price only) beat inflation:", yearly["Beat_inflation"].sum(), "of", len(yearly))

yearly[["SP500_return", "Inflation"]].plot(kind="bar", figsize=(9, 4),
                                           title="Calendar-year S&P 500 price return vs CPI inflation")
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

**Caveat:** `^GSPC` tracks prices only. Dividend yields were roughly 3% to 5% in the 1970s, so leaving them out understates what an investor actually earned. The next steps add dividends back using Shiller's data.

In [ ]:
# Step 9: Shiller data (price, annualized dividend, CPI), monthly
shiller = pd.read_csv("https://raw.githubusercontent.com/datasets/s-and-p-500/main/data/data.csv",
                      index_col=0, parse_dates=True)
shiller.index = shiller.index.to_period("M")
sh = shiller.loc["1969-12":"1979-12", ["SP500", "Dividend", "Consumer Price Index"]]
sh.columns = ["SP500", "Dividend", "CPI"]
display(sh.head())
display(sh.tail())

In [ ]:
# Step 10: monthly total return = (price today + 1/12 of annual dividend) / price last month - 1
sh["TR"] = (sh["SP500"] + sh["Dividend"] / 12) / sh["SP500"].shift(1) - 1
sh["TR_index"] = (1 + sh["TR"].fillna(0)).cumprod() * 100
sh["CPI_index"] = sh["CPI"] / sh["CPI"].iloc[0] * 100
display(sh.tail())

sh[["TR_index", "CPI_index"]].plot(figsize=(9, 4),
    title="S&P 500 total return (with dividends) vs CPI, Dec 1969 = 100")
plt.ylabel("Index level")
plt.show()

In [ ]:
# Step 11: annualized total return vs inflation
tr_ann = (sh["TR_index"].iloc[-1] / 100) ** (1 / years) - 1
cpi_ann = (sh["CPI_index"].iloc[-1] / 100) ** (1 / years) - 1
real_tr_ann = (1 + tr_ann) / (1 + cpi_ann) - 1
print(f"S&P 500 total return, annualized: {tr_ann:.2%}")
print(f"CPI inflation, annualized:        {cpi_ann:.2%}")
print(f"Real total return, annualized:    {real_tr_ann:.2%}")
print(f"Real value of $100 invested Dec 1969, at Dec 1979: ${sh['TR_index'].iloc[-1] / sh['CPI_index'].iloc[-1] * 100:.2f}")

## Conclusion
No. Over the 1970s (Dec 1969 to Dec 1979), U.S. stocks did not beat inflation.

- **Price only (Yahoo `^GSPC`):** the S&P 500 went from 92.06 to 107.94, up 17.2% in total (1.60% a year). CPI (FRED `CPIAUCSL`) went from 37.7 to 76.9, up 104.0% (7.39% a year). In real terms the index lost 42.5% (-5.39% a year).
- **With dividends reinvested (Shiller data):** total return was 5.84% a year against 7.36% inflation, a real return of -1.41% a year. $100 invested at the end of 1969 was worth about $87 in 1969 dollars by the end of 1979.
- **Year by year:** price returns beat inflation in only 4 of 10 years (1971, 1972, 1975, 1976). The 1973 to 1974 bear market (-17.4% and -29.7%) hit while inflation ran at 9% to 12%, and the decade never recovered from it.

**Limitations:** `^GSPC` ignores dividends, so the price-only result overstates the shortfall; the Shiller extension fixes that. Shiller's prices are monthly averages, not month-end closes, so his figures differ slightly from Yahoo's. Taxes and trading costs are ignored, and both would make the real result worse.